### Automatische Anpassung der Statistics.md des Git-Repositories

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd
import datetime
import os

# Pfade definieren
ASSETS_DIR = cfg.paths.assets_dir
DATA_DIR = cfg.paths.data_dir
OUTPUT_FILE = cfg.asset_path("statistics_output")

# 1. Lade die Evaluationstabelle & Performance Summary (als String)
table_path = os.path.join(cfg.asset_path("evaluation_table"))
print(table_path)
with open(table_path, "r", encoding="utf-8") as f:
    evaluation_table_md = f.read()
summary_path = os.path.join(ASSETS_DIR, cfg.asset_path("performance_summary"))
with open(summary_path, "r", encoding="utf-8") as f:
    performance_summary_md = f.read()
with open(os.path.join(ASSETS_DIR, cfg.asset_path("sorr_summary")), "r", encoding="utf-8") as f:
    sorr_summary_md = f.read()
with open(os.path.join(ASSETS_DIR, cfg.asset_path("mcs_summary")), "r", encoding="utf-8") as f:
    mcs_summary_md = f.read()
# Pipeline-Timing laden
timing_path = os.path.join(ASSETS_DIR, cfg.asset_path("pipeline_timing"))
try:
    with open(timing_path, "r", encoding="utf-8") as f:
        pipeline_timing_md = f.read()
except FileNotFoundError:
    pipeline_timing_md = "*Keine Timing-Daten verfügbar.*"

# Fast Mode Status aus config auslesen
fast_mode_enabled = cfg.fast_mode.enabled
fast_mode_status = "TRUE (Development Mode)" if fast_mode_enabled else "FALSE (Full Run)"

# Model Persistence Status aus config auslesen
persist_enabled = cfg.model_persistence.enabled
persist_dir = cfg.model_persistence.models_dir
# Pro Modell prüfen, ob ein persistiertes Modell geladen wurde
model_status_rows = []
for key in ['msm', 'hmm', 'lstm', 'transformer']:
    filename = getattr(cfg.model_persistence.files, key)
    filepath = os.path.join(persist_dir, filename)
    file_exists = os.path.exists(filepath)
    if persist_enabled and file_exists:
        status = "Geladen (persistiert)"
    else:
        status = "Neu trainiert"
    model_status_rows.append(f"| {key.upper()} | `{filename}` | {status} |")
model_persistence_table = "\n".join([
    "| Modell | Datei | Status |",
    "|:---|:---|:---|",
] + model_status_rows)
persist_status_text = "AKTIV" if persist_enabled else "DEAKTIVIERT"

# 2. Erstelle den dynamischen Inhalt
timestamp = datetime.datetime.now().strftime("%d.%m.%Y %H:%M")

stats_md_content = f"""
# Detaillierte statistische Auswertung & Forschungsergebnisse

Diese Seite dokumentiert die numerischen und grafischen Ergebnisse der Forschungs-Pipeline. Alle Auswertungen basieren auf dem Datensatz bis zum gestrigen Tag und werden automatisiert aktualisiert.

---

## 1. Executive Summary: Performance & Risiko
Ein direkter Vergleich der Kernkennzahlen über den gesamten **Out-of-Sample Testzeitraum**.

{performance_summary_md}

> **Kernaussage:** Vergleiche den **Max Drawdown** der aktiven Strategien mit der Buy & Hold Benchmark. Ziel der Arbeit ist eine signifikante Reduktion dieses Werts zur Minderung des SORR.

---

## 2. Datenbasis & Baseline Portfolio
Grundlage der Untersuchung ist ein globaler Multi-Asset-Ansatz.

### 60/40 Portfolio Kapitalkurve
Die Abbildung zeigt die kumulierte Wertentwicklung des statischen Referenzportfolios (60% Aktien / 40% Anleihen).

![Capital Curve](../assets/{cfg.paths.assets.capital_curve})

*   **Datenquelle:** S&P 500 (`^GSPC`) und Vanguard Long-Term Treasury (`VUSTX`).
*   **Reproduzierbarkeit:** Der bereinigte Datensatz inkl. aller Features ist hinterlegt unter: `data/02_feature_engineered_data.parquet`.

---

## 3. Regime-Erkennung der Einzelmodelle
Hier werden die Identifikations-Ergebnisse der Modell-Kategorien (Statistik, Clustering, Deep Learning) visualisiert.

### A. Markov-Switching-Modelle (Ökonometrie)
Identifikation von Bull- und Bear-Regimes mittels eines univariaten Zwei-Regime-Markov-Switching-Modells auf Basis der S&P 500-Renditen.
![Markov Models](../assets/{cfg.paths.assets.markov_model})

### B. Hidden Markov Model (Unsupervised Clustering)
![HMM Regimes](../assets/{cfg.paths.assets.hmm_regimes})

### C. LSTM-Netzwerk (Deep Learning)
Vorhersage der Marktphasen durch das neuronale Netzwerk (trainiert auf Markov-Labels).
![LSTM Model](../assets/{cfg.paths.assets.lstm_model})

### D. Transformer-Netzwerk (Attention-basierte Regime-Erkennung)
"Klassifikation von Marktregimes mittels eines Transformer-Encoders mit Multi-Head Self-Attention und Positional Encoding. Im Gegensatz zu rekurrenten Architekturen (LSTM) verarbeitet der Transformer alle Zeitschritte einer Sequenz parallel und lernt über den Attention-Mechanismus, welche historischen Datenpunkte die höchste Relevanz für die aktuelle Regime-Klassifikation besitzen. Trainiert im Supervised-Setting auf Markov-Labels.
![Transformer Model](../assets/{cfg.paths.assets.transformer_model})

### E. Globaler Regime-Vergleich
Detaillierte Gegenüberstellung der Wahrscheinlichkeiten und harten Signale aller Modelle.
![Regime Comparison](../assets/{cfg.paths.assets.regime_comparison})

---

## 4. Backtesting & Strategie-Evaluation
Die ökonomische Anwendung der Regime-Signale durch dynamische Umschichtung in den Geldmarkt.

### Equity Curves im Vergleich
![Equity Curves](../assets/{cfg.paths.assets.equity_curves})

### Umfassende Kennzahlen-Matrix
Detaillierte statistische Analyse inklusive risikoadjustierter Kennzahlen (Sharpe, Sortino, Calmar).

{evaluation_table_md}

### Transaktionskosten

Diese Grafik zeigt die kumulierten Transaktionskosten im Zeitverlauf. Steile Anstiege deuten auf instabile Regime-Wechsel ("Churning") hin.

![Transaction Costs](../assets/{cfg.paths.assets.transaction_costs})

Stress-Test: Sequence of Returns Risk (SORR)
Außerdem wurde die Überlebensdauer des Kapitals in einer simulierten Entnahmephase (Ruhestandsszenario) durchgeführt.

### SORR-Simulation: Vergleich der Entnahmeszenarien

In dieser Tabelle werden verschiedene Stress-Szenarien (Standard, Aggressiv, Geringes Kapital) gegenübergestellt.

{sorr_summary_md}

Abbildung der Kapitalentwicklung der unterschiedlichen Szenarien:
![SORR Standard](../assets/{cfg.paths.assets.sorr_sim_standard})
![SORR Aggressive](../assets/{cfg.paths.assets.sorr_sim_aggressive})
![SORR Low Capital](../assets/{cfg.paths.assets.sorr_sim_low_capital})

### MCS: Block-Bootstrap Robustness-Check

Um die statistische Signifikanz zu prüfen, wurden 1.000 künstliche Marktpfade mittels Block-Bootstrap simuliert.
![MCS Paths](../assets/{cfg.paths.assets.mcs_paths})
{mcs_summary_md}

Verteilung der Endkapitalwerte:

![MCS Boxplots Standard](../assets/{cfg.paths.assets.mcs_boxplot_standard})
![MCS Boxplots Aggressive](../assets/{cfg.paths.assets.mcs_boxplot_aggressive})
![MCS Boxplots Low Capital](../assets/{cfg.paths.assets.mcs_boxplot_low_capital})

Wahrscheinlichkeitskorridore:

Die schattierten Bereiche zeigen das 5% bis 95% Konfidenzintervall der Kapitalentwicklung.
![MCS Quantiles](../assets/{cfg.paths.assets.mcs_quantiles})

---

## Forschungsnotizen & Methodik
- **Cash-Komponente:** Bei einem "Bear"-Signal schichtet die Strategie in den aktuellen Geldmarktzins (**^IRX**) um.
- **Vermeidung von Look-ahead Bias:** Alle Signale werden für das Backtesting um einen Tag zeitversetzt (`shift(1)`), um reale Handelsbedingungen zu simulieren.
- **Feature-Set:** Die Modelle nutzen Renditen, Volatilität (20d), SMA-Abstand, Momentum, VIX und Yield Spread.
- **Kostensimulation:** Es wird eine pauschale Gebühr von 10 Basispunkten (0,1%) pro Umschichtung berechnet.
- **SORR-Spezifika:** Bei Entnahmen in "Bull"-Phasen wird eine zusätzliche Liquiditätsgebühr von 0,1% auf den Entnahmebetrag erhoben (Asset-Verkäufe). In "Bear"-Phasen (Cash) entfällt diese.

---

## Pipeline-Laufzeiten

Ausführungszeiten der einzelnen Pipeline-Notebooks (monolithischer Notebook-Ansatz).

{pipeline_timing_md}

---

## Modell-Persistierung

Status der Modell-Persistierung für diesen Pipeline-Durchlauf:

- **Persistierung:** {persist_status_text}
- **Modell-Verzeichnis:** `{persist_dir}`

{model_persistence_table}

> **Hinweis:** Bei aktivierter Persistierung werden vortrainierte Modelle aus `{persist_dir}` geladen, sofern die Dateien existieren. Andernfalls wird normal trainiert und das Ergebnis für zukünftige Läufe gespeichert. Bei Änderungen an Hyperparametern müssen die entsprechenden Modelldateien gelöscht werden.

---

**Zuletzt aktualisiert:** {timestamp}<br>
**Fast Mode Status zur Laufzeit:** {fast_mode_status}<br>
**Modell-Persistierung:** {persist_status_text}<br>
*Generiert durch die automatisierte ETL-Pipeline (Notebook 99).*
"""

# 3. Datei schreiben
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(stats_md_content)

print(f"Statistics.md wurde erfolgreich aktualisiert ({timestamp}).")